In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PidMLP import PidMLP

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
action_space_n = 50
action_bins = np.linspace(-2, 2, action_space_n)  # Define bins for discretization

In [3]:
# Load data
df = pd.read_csv('pid_controller_data.csv')

# Seperate features and labels
features = df[['target_lataccel', 'current_lataccel', 'vEgo', 'aEgo', 'roll', 'error', 'prev_error', 'integral', 'derivative', 'prev_derivative', "prev_action"]].values
labels = df['steerCommand'].values

# Discretize labels and one-hot encode
bin_indices = np.digitize(labels, action_bins, right=True) - 1
bin_indices = np.clip(bin_indices, 0, action_space_n - 1)

# Convert to tensors
features_tensor = torch.tensor(features, dtype=torch.float32, device=device)
labels_tensor = torch.tensor(bin_indices, dtype=torch.long, device=device)

# Create a TensorDataset
dataset = TensorDataset(features_tensor, labels_tensor)

# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=2048, shuffle=True)

In [9]:
# Calculate the averages
average_prev_error = df['prev_error'].mean()
average_prev_derivative = df['prev_derivative'].mean()
average_integral = df['integral'].mean()
average_prev_action = df['prev_action'].mean()

# Print the averages
print(f"Average of prev_error: {average_prev_error}")
print(f"Average of prev_derivative: {average_prev_derivative}")
print(f"Average of integral: {average_integral}")
print(f"Average of prev_action: {average_prev_action}")

Average of prev_error: -0.005545284143109326
Average of prev_derivative: -8.187370369147938e-06
Average of integral: -2.083133838593108
Average of prev_action: -0.06360434591448254


In [4]:
len(dataloader)

609

In [5]:
# Instantiate model
model = PidMLP(in_features=11, output_space_n=action_space_n, dropout_rate=0.3).to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
schedular = optim.lr_scheduler.StepLR(optimizer, 5, 0.5)

# Training loop
epochs = 20
losses = []
for epoch in range(epochs):
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        
        # Calculate loss
        loss = criterion(outputs, targets)
        total_loss += loss.item()
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_predictions += targets.size(0)
        correct_predictions += (predicted == targets).sum().item()

    schedular.step()

    # Compute average loss
    average_loss = total_loss / len(dataloader)
    losses.append(average_loss)

    # Compute accuracy
    accuracy = (correct_predictions / total_predictions) * 100
    
    # Print loss and accuracy
    print(f'End of Epoch {epoch+1}, Loss: {average_loss:.4f}, Accuracy: {accuracy:.2f}%')

End of Epoch 1, Loss: 0.8908, Accuracy: 66.66%
End of Epoch 2, Loss: 0.3763, Accuracy: 85.33%
End of Epoch 3, Loss: 0.2790, Accuracy: 89.29%
End of Epoch 4, Loss: 0.2307, Accuracy: 91.26%
End of Epoch 5, Loss: 0.1995, Accuracy: 92.50%
End of Epoch 6, Loss: 0.1738, Accuracy: 93.45%
End of Epoch 7, Loss: 0.1631, Accuracy: 93.88%
End of Epoch 8, Loss: 0.1561, Accuracy: 94.17%
End of Epoch 9, Loss: 0.1478, Accuracy: 94.47%
End of Epoch 10, Loss: 0.1432, Accuracy: 94.66%
End of Epoch 11, Loss: 0.1306, Accuracy: 95.15%
End of Epoch 12, Loss: 0.1272, Accuracy: 95.24%
End of Epoch 13, Loss: 0.1241, Accuracy: 95.38%
End of Epoch 14, Loss: 0.1213, Accuracy: 95.48%
End of Epoch 15, Loss: 0.1187, Accuracy: 95.56%
End of Epoch 16, Loss: 0.1132, Accuracy: 95.77%
End of Epoch 17, Loss: 0.1107, Accuracy: 95.85%
End of Epoch 18, Loss: 0.1099, Accuracy: 95.91%
End of Epoch 19, Loss: 0.1091, Accuracy: 95.94%
End of Epoch 20, Loss: 0.1074, Accuracy: 95.98%


In [3]:
model = torch.load('./models/mlp_pid.pth')

In [6]:
torch.save(model, './models/mlp_pid.pth')

In [7]:
model.eval()
action = model(torch.tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device=device, dtype=torch.float32))
print(action)

tensor([-10.1394, -14.8023, -16.3602, -17.0029, -18.8699, -19.1780, -20.2664,
        -17.6791, -18.1063, -18.3174, -19.0813, -19.9018, -17.8793, -17.0813,
        -16.3199, -15.0130, -12.7703, -10.4727, -11.3676, -15.4759, -21.8626,
        -25.0146, -20.4860, -18.8426, -16.1516, -12.4956,  -4.3008,   4.1402,
          9.1302,   6.1117,   0.2688,  -3.0252,  -5.0693,  -6.2648,  -7.9869,
         -8.9775, -10.5711, -12.2796, -13.8317, -15.8072, -17.6597, -18.5959,
        -19.2134, -19.6244, -22.1486, -20.6060, -15.5058, -10.7044, -10.5877,
        -10.4123], device='cuda:0', grad_fn=<ViewBackward0>)
